In [1]:
import cv2
import os
import re
import numpy as np
from ultralytics import YOLO
import easyocr
from collections import Counter
from pathlib import Path
from collections import defaultdict
import pandas as pd
from yt_dlp import YoutubeDL
import imageio_ffmpeg


In [2]:
# Scoreboard! 
YOUTUBE_LINK = "https://www.youtube.com/watch?v=NSWVoO4ZDEs" # video link to process
VIDEOS_PATH = "videos" # The directory where the downloaded videos are stored.

CROP_MODEL_PATH = "/work/classtmp/ryanbog/robots/FIRST-Robotics-Competition-Data-Challenge/models/crop_scoreboard.pt"
INFO_MODEL_PATH = "/work/classtmp/ryanbog/robots/FIRST-Robotics-Competition-Data-Challenge/models/extract_scoreboard_info.pt" 

FRAME_SKIP = 15 # How many frames are skipped between each processing step. If the value is 15, it will process 1 frame every 15 frames.

DEVICE = 0  # This variable specifies what GPU(s) you use (if available). Can be set to "cpu", 0, [0,1], etc.

DELETE_VIDEO = False # Whether to delete the downloaded video after processing to save space. Set to True to enable deletion.

# Initialize directories and models
os.makedirs(VIDEOS_PATH, exist_ok=True)

crop_model = YOLO(CROP_MODEL_PATH)
info_model = YOLO(INFO_MODEL_PATH)

reader = easyocr.Reader(['en'], gpu=(DEVICE != "cpu"))

# Used to enhance the region of interest (ROI) for better OCR performance. It resizes, increases contrast, and applies thresholding.
def preprocess_for_ocr(roi):
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)
    gray = cv2.convertScaleAbs(gray, alpha=1.8, beta=10)

    _, thresh = cv2.threshold(
        gray, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    thresh_inv = cv2.bitwise_not(thresh)
    return [thresh, thresh_inv]

# Runs OCR on multiple images and returns the highest-confidence text
def read_best_text(images, allowlist):
    best_text, best_conf = None, 0

    for img in images:
        results = reader.readtext(
            img,
            allowlist=allowlist,
            detail=1,
            paragraph=False
        )
        for (_, text, conf) in results:
            if conf > best_conf:
                best_conf, best_text = conf, text

    return best_text

# Extracts the numeric value from the image using OCR
def read_number(img):
    images = preprocess_for_ocr(img)
    text = read_best_text(images, '0123456789')

    if text is None:
        return None

    text = re.sub(r"\D", "", text)
    return int(text) if text else None

# Extracts the timer value in MM:SS format from the image using OCR
def read_timer(img):
    images = preprocess_for_ocr(img)
    text = read_best_text(images, '0123456789:')

    if text is None:
        return None

    match = re.search(r"\d{1,2}:\d{2}", text)
    return match.group(0) if match else None

# Download video at best quality and return the saved filename
def download_video(url, output_dir):
    ydl_opts = {
        "format": "bestvideo+bestaudio/best",
        "outtmpl": os.path.join(output_dir, "%(title)s.%(ext)s"),
        "ffmpeg_location": imageio_ffmpeg.get_ffmpeg_exe(),
        "merge_output_format": "mp4",
    }

    with YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
        filename = ydl.prepare_filename(info)
        return os.path.splitext(os.path.basename(filename))[0] + ".mp4"

# Download the video
video_filename = download_video(YOUTUBE_LINK, VIDEOS_PATH)
video_path = os.path.join(VIDEOS_PATH, video_filename)

# Storage for final rows and tracking previous scores
rows = []
prev_blue = None
prev_red = None
pending_row = None # Holds the first occurence of a score change when the timer is missing
is_auto = True

# Extract the youtube url for this video
youtube_url = YOUTUBE_LINK

cap = cv2.VideoCapture(video_path)
frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Skip frames according to FRAME_SKIP
    if frame_idx % FRAME_SKIP != 0:
        frame_idx += 1
        continue
    
    # Detect the scoreboard region
    crop_results = crop_model(frame, device=DEVICE)[0]

    # Skip if no scoreboard is found
    if len(crop_results.boxes) == 0:
        frame_idx += 1
        continue

    # Crop scoreboard from frame
    x1, y1, x2, y2 = map(int, crop_results.boxes.xyxy[0])
    scoreboard = frame[y1:y2, x1:x2]

    # Find the elements inside the cropped frame (scores, timer, team numbers)
    info_results = info_model(scoreboard, device=DEVICE)[0]

    # Initialize variables for this frame
    blue_score = None
    red_score = None
    timer = None


    blue_center_x = None
    red_center_x = None

    team_data = []

    # Loop through each of the detected elements
    for b in info_results.boxes:

        cls_id = int(b.cls[0])
        label = info_model.names[cls_id]

        x1, y1, x2, y2 = map(int, b.xyxy[0])
        region = scoreboard[y1:y2, x1:x2]

        x_center = (x1 + x2) / 2

        # Read blue score
        if label == "blue_score":
            blue_score = read_number(region)
            blue_center_x = x_center

        # Read red score
        elif label == "red_score":
            red_score = read_number(region)
            red_center_x = x_center

        # Read timer
        elif label == "timer":
            timer = read_timer(region)

        # Read team number
        elif label == "team_number":
            num = read_number(region)
            if num is not None:
                team_data.append((num, x_center))

    # If both scores and timer are missing, skip the frame
    if blue_score is None and red_score is None and timer is None:
        frame_idx += 1
        continue

    # Ensure blue score doesn't decrease compared to previous frames
    if prev_blue is not None and blue_score is not None:
        if blue_score < prev_blue:
            frame_idx += 1
            continue

    # Ensure red score doesn't decrease compared to previous frames
    if prev_red is not None and red_score is not None:
        if red_score < prev_red:
            frame_idx += 1
            continue

    # If the scores haven't changed, skip the frame
    if prev_blue == blue_score and prev_red == red_score:
        frame_idx += 1
        continue

    # Determine if it is still auto stage or not
    if timer is not None and is_auto:
        try:
            minutes = int(timer.split(":")[0])
            if minutes == 2:
                is_auto = False
        except:
            pass

    # Identify if the red score is on the left or right side of the scoreboard
    red_location = None
    if blue_center_x is not None and red_center_x is not None:
        red_location = "right" if red_center_x > blue_center_x else "left"

    # Split the team numbers into blue vs red based on their loation to the score
    blue_teams = []
    red_teams = []

    midpoint = (blue_center_x + red_center_x) / 2 if blue_center_x and red_center_x else None

    if midpoint:
        for num, x in team_data:
            if red_location == "right":
                if x < midpoint:
                    blue_teams.append((num, abs(x - blue_center_x)))
                else:
                    red_teams.append((num, abs(x - red_center_x)))
            else:
                if x > midpoint:
                    blue_teams.append((num, abs(x - blue_center_x)))
                else:
                    red_teams.append((num, abs(x - red_center_x)))

    # Keep the closest 3 teams for each alliance
    blue_team_numbers = [n for n, _ in sorted(blue_teams, key=lambda x: x[1])[:3]]
    red_team_numbers = [n for n, _ in sorted(red_teams, key=lambda x: x[1])[:3]]

    # Build row
    current_row = {
        "Frame": frame_idx,
        "blue_score": blue_score,
        "red_score": red_score,
        "timer": timer,
        "is_auto": is_auto,
        "red_location": red_location,
        "blue_team_numbers": blue_team_numbers,
        "red_team_numbers": red_team_numbers,
        "youtube_link": youtube_url,
    }

    # If the timer is missing, store the very first instance for this score change
    if blue_score is not None and red_score is not None and timer is None:
        if pending_row is None:
            pending_row = current_row
        frame_idx += 1
        continue

    # If the timer ends up appearing in a later frame with the same scores, use that row instead
    if (
        pending_row is not None and
        timer is not None and
        blue_score == pending_row["blue_score"] and
        red_score == pending_row["red_score"]
    ):
        rows.append(current_row)
        pending_row = None

    # If score changes, flush the pending row and add the current row
    else:
        if pending_row is not None:
            rows.append(pending_row)
            pending_row = None

        rows.append(current_row)

    # Update previous scores
    prev_blue = blue_score
    prev_red = red_score

    frame_idx += 1

# Save any remaining pending row at the end of the video
if pending_row is not None:
    rows.append(pending_row)
    pending_row = None

cap.release()

# Delete video after processing to save space
if DELETE_VIDEO == True and os.path.exists(video_path):
    os.remove(video_path)

# Convert to dataframe and display
output_df = pd.DataFrame(rows)
output_df

[youtube] Extracting URL: https://www.youtube.com/watch?v=NSWVoO4ZDEs
[youtube] NSWVoO4ZDEs: Downloading webpage


[youtube] NSWVoO4ZDEs: Downloading android vr player API JSON
[info] NSWVoO4ZDEs: Downloading 1 format(s): 137+251
[download] Destination: videos/Final 1 - 2025 Rocket City Regional.f137.mp4
[download] 100% of   91.87MiB in 00:00:02 at 38.01MiB/s    
[download] Destination: videos/Final 1 - 2025 Rocket City Regional.f251.webm
[download] 100% of    2.30MiB in 00:00:00 at 8.92MiB/s   
[Merger] Merging formats into "videos/Final 1 - 2025 Rocket City Regional.mp4"
Deleting original file videos/Final 1 - 2025 Rocket City Regional.f137.mp4 (pass -k to keep)
Deleting original file videos/Final 1 - 2025 Rocket City Regional.f251.webm (pass -k to keep)

0: 384x640 (no detections), 42.7ms
Speed: 5.0ms preprocess, 42.7ms inference, 3.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 6.8ms
Speed: 1.7ms preprocess, 6.8ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 6.8ms
Speed: 1.4ms preprocess, 6.8ms inference, 0.4ms p

,Frame,blue_score,red_score,timer,is_auto,red_location,blue_team_numbers,red_team_numbers,youtube_link
0,105,0,0,0:00,True,right,"[10011, 4020, 2338]","[2783, 4028, 7111]",https://www.youtube.com/watch?v=NSWVoO4ZDEs
1,315,9,0,0:13,True,right,"[10011, 4020, 2338]","[2783, 4028, 7111]",https://www.youtube.com/watch?v=NSWVoO4ZDEs
2,345,9,7,0:12,True,right,"[10011, 4020, 2338]","[2783, 4028, 7111]",https://www.youtube.com/watch?v=NSWVoO4ZDEs
3,360,16,14,0:11,True,right,"[10011, 4020, 2338]","[2783, 4028, 7111]",https://www.youtube.com/watch?v=NSWVoO4ZDEs
4,375,23,17,0:11,True,right,"[10011, 4020, 2338]","[2783, 4028, 7111]",https://www.youtube.com/watch?v=NSWVoO4ZDEs
5,390,23,23,0:10,True,right,"[10011, 4020, 2338]","[2783, 4028, 7111]",https://www.youtube.com/watch?v=NSWVoO4ZDEs
6,495,30,23,0:07,True,right,"[10011, 4020, 2338]","[2783, 4028, 7111]",https://www.youtube.com/watch?v=NSWVoO4ZDEs
7,510,37,23,0:06,True,right,"[10011, 4020, 2338]","[2783, 4028, 7111]",https://www.youtube.com/watch?v=NSWVoO4ZDEs
8,570,37,28,0:04,True,right,"[10011, 4020, 2338]","[2783, 4028, 7111]",https://www.youtube.com/watch?v=NSWVoO4ZDEs
9,585,37,37,0:04,True,right,"[10011, 4020, 2338]","[2783, 4028, 7111]",https://www.youtube.com/watch?v=NSWVoO4ZDEs


In [3]:
# CONFIG
REPO_ROOT = Path(os.getcwd()).parent  

VIDEO_NAME = 'cropped_Qualification 45 - 2025 Central Missouri Regional.mp4'
VIDEO_PATH = "/work/classtmp/FIRST-Robotics-Competition-Data-Challenge-Videos/cropped_videos/cropped_Qualification 45 - 2025 Central Missouri Regional.mp4"

ROBOT_MODEL_PATH = REPO_ROOT / "yolov8_model" / "best_tuned_yolov8.pt"
# NUMBER_MODEL_PATH = REPO_ROOT / "number_reading" / "best_number.pt"

# Changed because of different file names 
NUMBER_MODEL_PATH = REPO_ROOT / "robot_numbers" / "runs" / "detect" / "train" / "weights" / "best.pt"

ROBOT_CLASS_ID = 1
REEF_CLASS_ID = 0
BLUE_NUMBER_CLASS_ID = 0
RED_NUMBER_CLASS_ID = 1

FRAME_SKIP = 15

# Custom tracker 
CUSTOM_TRACKER_PATH = REPO_ROOT / "trackers" / "botsort_custom.yaml"


In [4]:
# LOAD MODELS
robot_model = YOLO(ROBOT_MODEL_PATH)
number_model = YOLO(NUMBER_MODEL_PATH)

reader = easyocr.Reader(['en'], gpu=True)

In [5]:
# BOTSORT
from collections import defaultdict

print("Loading model...")
model = YOLO(ROBOT_MODEL_PATH)

def run_botsort(model_path, video_path, tracker_path, save_video=False, output_dir=None, stride=1):

    print("Running BotSort tracking...")

    results = model.track(
        source=video_path,
        tracker=tracker_path,
        stream=True,
        persist=True,
        conf=0.35,
        device=0,
        vid_stride=stride,
        verbose=False
    )

    tracking_results = []
    reef_results = []

    frame_data = defaultdict(lambda: {"robots": [], "reef": None})

    video_writer = None

    for frame_i, result in enumerate(results):

        if result.boxes is None or result.boxes.xyxy is None:
            continue

        boxes = result.boxes.xyxy.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy().astype(int)
        ids = result.boxes.id

        if ids is None:
            continue

        ids = ids.cpu().numpy().astype(int)

        frame = result.plot()

        for i in range(len(boxes)):

            x1, y1, x2, y2 = boxes[i]
            cls = classes[i]
            track_id = ids[i]

            if cls == ROBOT_CLASS_ID:

                tracking_results.append({
                    "frame": frame_i * stride,
                    "track_id": int(track_id),
                    "box": [float(x1), float(y1), float(x2), float(y2)],
                    "crop": frame[int(y1):int(y2), int(x1):int(x2)].copy()
                })

                frame_data[frame_i * stride]["robots"].append({
                    "track_id": int(track_id),
                    "bbox": [float(x1), float(y1), float(x2), float(y2)]
                })

                cv2.putText(
                    frame,
                    f"ID {track_id}",
                    (int(x1), int(y1) - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (0, 255, 0),
                    2
                )

            elif cls == REEF_CLASS_ID:
                reef_results.append({
                    "frame": frame_i * stride,
                    "box": [float(x1), float(y1), float(x2), float(y2)]
            })

        if save_video:
            if video_writer is None:
                h, w = frame.shape[:2]
                output_path = output_dir / "botsort_with_ids.mp4"

                video_writer = cv2.VideoWriter(
                    str(output_path),
                    cv2.VideoWriter_fourcc(*"mp4v"),
                    30,
                    (w, h)
                )

            video_writer.write(frame)

    if video_writer:
        video_writer.release()

    print("Tracking complete.")
    
    return tracking_results, frame_data, reef_results

Loading model...


In [6]:
# IMAGE PROCESSING FUNCTIONS
def estimate_angle_from_crop(crop):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 180, 255, cv2.THRESH_BINARY)

    contours, _ = cv2.findContours(
        thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    if not contours:
        return 0.0

    cnt = max(contours, key=cv2.contourArea)
    rect = cv2.minAreaRect(cnt)
    angle = rect[-1]

    if angle < -45:
        angle += 90

    return angle


def rotate_image(img, angle):
    h, w = img.shape[:2]
    center = (w // 2, h // 2)

    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(
        img,
        M,
        (w, h),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_REPLICATE
    )


def preprocess_crop(crop, threshold):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)

    angle = estimate_angle_from_crop(crop)
    rotated = rotate_image(thresh, angle)

    return rotated

def preprocess_variants(crop):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

    variants = []

    # 1. Simple thresholds
    for t in [150, 165, 180, 195, 210]:
        _, th = cv2.threshold(gray, t, 255, cv2.THRESH_BINARY)
        variants.append(th)

    # 2. Adaptive threshold
    adaptive = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        11, 2
    )
    variants.append(adaptive)

    # 3. Otsu
    _, otsu = cv2.threshold(
        gray, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )
    variants.append(otsu)

    # 4. Histogram equalization + Otsu
    eq = cv2.equalizeHist(gray)
    _, th_eq = cv2.threshold(eq, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    variants.append(th_eq)

    return variants

In [7]:
# OCR
def read_number_from_image(img):
    variants = preprocess_variants(img)
    guesses = []

    angle = estimate_angle_from_crop(img)
    for processed in variants:
        rotated = rotate_image(processed, angle)

        results = reader.readtext(
            rotated,
            allowlist="0123456789",
            detail=1,
            paragraph=False
        )

        # Filter low confidence
        results = [r for r in results if r[2] > 0.5]

        if results:
            guesses.append(results[0][1])

    if not guesses:
        return None

    c = Counter(guesses)
    max_count = max(c.values())

    tied = [num for num, count in c.items() if count == max_count]

    return max(tied, key=lambda x: len(str(x)))

In [8]:
# MATCHING SYSTEM
def levenshtein(a, b):
    a, b = str(a), str(b)
    dp = [[0]*(len(b)+1) for _ in range(len(a)+1)]

    for i in range(len(a)+1):
        dp[i][0] = i
    for j in range(len(b)+1):
        dp[0][j] = j

    for i in range(1, len(a)+1):
        for j in range(1, len(b)+1):
            cost = 0 if a[i-1] == b[j-1] else 1
            dp[i][j] = min(
                dp[i-1][j] + 1,
                dp[i][j-1] + 1,
                dp[i-1][j-1] + cost
            )

    return dp[-1][-1]


def similarity(a, b):
    if not a or not b:
        return 0
    dist = levenshtein(a, b)
    return 1 - dist / max(len(str(a)), len(str(b)))


def alignment_score(a, b):
    a, b = str(a), str(b)
    best = 0

    for shift in range(-len(b), len(a)+1):
        matches = 0
        for i in range(len(a)):
            j = i - shift
            if 0 <= j < len(b) and a[i] == b[j]:
                matches += 1
        best = max(best, matches)

    return best / max(len(a), len(b))


def combined_score(a, b, w1=0.7, w2=0.3):
    return w1 * similarity(a, b) + w2 * alignment_score(a, b)


def match_number_single(detected, nums):
    if detected is None:
        return None

    best_score = 0.5
    match = None

    for n in nums:
        current = combined_score(detected, n)
        if current > best_score:
            best_score = current
            match = n

    return match

In [9]:
def apply_elimination(final_labels, track_candidates, all_ids, tracks):
    assigned_ids = set(final_labels.values())
    remaining_ids = set(all_ids) - assigned_ids

    # Tracks with no label yet
    unlabeled_tracks = [t for t in tracks if t not in final_labels]

    # Case 1: Perfect elimination
    if len(unlabeled_tracks) == len(remaining_ids):
        for t, rid in zip(unlabeled_tracks, remaining_ids):
            final_labels[t] = rid

    # Case 2: Candidate filtering
    else:
        for t in unlabeled_tracks:
            candidates = set(track_candidates.get(t, []))
            candidates -= assigned_ids

            if len(candidates) == 1:
                final_labels[t] = candidates.pop()

    return final_labels

In [10]:
# NUMBER READING
def read_numbers(tracking_results, blue_team_numbers=[], red_team_numbers=[], stride=1):

    results = []
    current_frame = None
    frame_data = []

    for entry in tracking_results:

        frame_idx = entry["frame"]
        robot_crop = entry["crop"]

        if frame_idx % stride != 0:
            continue

        # Detect frame change
        if current_frame is None:
            current_frame = frame_idx

        if frame_idx != current_frame:
            print(f"Frame {current_frame}: {frame_data}")
            results.append({
                "frame": current_frame,
                "detections": frame_data
            })
            frame_data = []
            current_frame = frame_idx

        if robot_crop is None or robot_crop.size == 0:
            continue

        number_results = number_model(robot_crop, verbose=False)[0]

        for nbox in number_results.boxes:
            cls_id = int(nbox.cls[0])
            if cls_id == RED_NUMBER_CLASS_ID:
                team_list = red_team_numbers
            elif cls_id == BLUE_NUMBER_CLASS_ID:
                team_list = blue_team_numbers
            else:
                continue

            if nbox.conf[0] < 0.4:
                continue

            nx1, ny1, nx2, ny2 = map(int, nbox.xyxy[0])

            # Padding
            pad = 5
            h2, w2 = robot_crop.shape[:2]
            nx1 = max(0, nx1 - pad)
            ny1 = max(0, ny1 - pad)
            nx2 = min(w2, nx2 + pad)
            ny2 = min(h2, ny2 + pad)

            # Filter tiny boxes
            if (nx2 - nx1) < 30 or (ny2 - ny1) < 15:
                continue

            number_crop = robot_crop[ny1:ny2, nx1:nx2]

            if number_crop.size == 0:
                continue

            detected = read_number_from_image(number_crop)
            matched = match_number_single(detected, team_list)

            frame_data.append({
                "detected": detected,
                "matched": matched,
                "track_id": entry["track_id"],
                "box": entry["box"],
                "alliance_hint": "blue" if cls_id == BLUE_NUMBER_CLASS_ID else "red"
            })

    # Add last frame
    if frame_data:
        print(f"Frame {current_frame}: {frame_data}")
        results.append({
            "frame": current_frame,
            "detections": frame_data
        })

    return results

In [11]:
def run_full_pipeline(
    video_path,
    robot_model_path,
    number_model_path,
    tracker_path,
    blue_ids,
    red_ids,
    output_path,
    tracking_stride=1,
    ocr_stride=1
):
    print("Running BotSort...")
    botsort_results = run_botsort(
        robot_model_path,
        video_path,
        tracker_path,
        stride=tracking_stride
    )

    print("Reading numbers...")
    output = read_numbers(
        botsort_results,
        blue_ids,
        red_ids,
        stride=ocr_stride
    )

    track_alliance_votes = defaultdict(lambda: {"blue": 0, "red": 0})

    for frame in output:
        for det in frame["detections"]:
            tid = det["track_id"]
            match = det["matched"]

            # Strong signal: detector class
            hint = det.get("alliance_hint")
            if hint == "blue":
                track_alliance_votes[tid]["blue"] += 2  
            elif hint == "red":
                track_alliance_votes[tid]["red"] += 2

            # Weak signal: OCR match
            if match in blue_ids:
                track_alliance_votes[tid]["blue"] += 1
            elif match in red_ids:
                track_alliance_votes[tid]["red"] += 1

    track_alliance = {}
    for tid, votes in track_alliance_votes.items():
        if votes["blue"] > votes["red"]:
            track_alliance[tid] = "blue"
        elif votes["red"] > votes["blue"]:
            track_alliance[tid] = "red"
        else:
            track_alliance[tid] = None  # uncertain

    track_memory = defaultdict(list)

    for frame in output:
        for det in frame["detections"]:
            tid = det["track_id"]
            match = det["matched"]

            if match is None:
                continue

            alliance = track_alliance.get(tid)

            # HARD FILTER: only accept correct alliance matches
            if alliance == "blue" and match in blue_ids:
                track_memory[tid].append(match)
            elif alliance == "red" and match in red_ids:
                track_memory[tid].append(match)

    final_labels = {}

    for tid, nums in track_memory.items():
        if len(nums) < 2:
            continue  # ignore weak tracks

        scores = defaultdict(float)

        # New: exponential decay weighting
        for i, num in enumerate(nums):
            weight = 0.9 ** (len(nums) - i)  # newer = higher weight
            scores[num] += weight

        best = max(scores, key=scores.get)

        # Optional safety check (prevents super noisy flips)
        total_weight = sum(scores.values())
        if scores[best] / total_weight >= 0.5:
            final_labels[tid] = best

    def eliminate(alliance_tracks, alliance_ids):
        assigned = set(
            v for k, v in final_labels.items()
            if track_alliance.get(k) in alliance_tracks
        )

        remaining_ids = set(alliance_ids) - assigned

        unlabeled_tracks = [
            t for t in track_alliance
            if track_alliance[t] in alliance_tracks and t not in final_labels
        ]

        if len(unlabeled_tracks) == len(remaining_ids):
            for t, rid in zip(unlabeled_tracks, remaining_ids):
                final_labels[t] = rid

    eliminate({"blue"}, blue_ids)
    eliminate({"red"}, red_ids)

    for tid, label in list(final_labels.items()):
        alliance = track_alliance.get(tid)

        if alliance == "blue" and label not in blue_ids:
            del final_labels[tid]
        elif alliance == "red" and label not in red_ids:
            del final_labels[tid]

    print("Final track labels:", final_labels)

    # -----------------------------
    # STEP 6: BUILD LOOKUP
    # -----------------------------
    tracking_lookup = defaultdict(list)

    for entry in botsort_results:
        tracking_lookup[entry["frame"]].append(entry)

    # -----------------------------
    # STEP 7: DRAW VIDEO
    # -----------------------------
    print("Drawing Video...")

    cap = cv2.VideoCapture(video_path)
    video_writer = None

    frame_idx = 0
    last_drawn = None

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_idx += 1

        if frame_idx in tracking_lookup:
            draw_frame = frame.copy()

            frame_tracks = tracking_lookup[frame_idx]

            blue_tracks_in_frame = []
            red_tracks_in_frame = []

            for entry in frame_tracks:
                tid = entry["track_id"]
                alliance = track_alliance.get(tid)

                if alliance == "blue":
                    blue_tracks_in_frame.append(tid)
                elif alliance == "red":
                    red_tracks_in_frame.append(tid)

            # --- BLUE elimination ---
            blue_labeled = {
                tid: final_labels[tid]
                for tid in blue_tracks_in_frame
                if tid in final_labels
            }

            blue_unlabeled = [
                tid for tid in blue_tracks_in_frame
                if tid not in final_labels
            ]

            remaining_blue_ids = set(blue_ids) - set(blue_labeled.values())

            if len(blue_unlabeled) == 1 and len(remaining_blue_ids) == 1:
                final_labels[blue_unlabeled[0]] = list(remaining_blue_ids)[0]

            # --- RED elimination ---
            red_labeled = {
                tid: final_labels[tid]
                for tid in red_tracks_in_frame
                if tid in final_labels
            }

            red_unlabeled = [
                tid for tid in red_tracks_in_frame
                if tid not in final_labels
            ]

            remaining_red_ids = set(red_ids) - set(red_labeled.values())

            if len(red_unlabeled) == 1 and len(remaining_red_ids) == 1:
                final_labels[red_unlabeled[0]] = list(remaining_red_ids)[0]


            for entry in tracking_lookup[frame_idx]:
                tid = entry["track_id"]
                x1, y1, x2, y2 = map(int, entry["box"])

                # Determine color based on alliance
                alliance = track_alliance.get(tid)

                if alliance == "blue":
                    color = (255, 0, 0)   # Blue (BGR)
                elif alliance == "red":
                    color = (0, 0, 255)   # Red (BGR)
                else:
                    color = (200, 200, 200)  # Unknown = gray

                # Draw box
                cv2.rectangle(draw_frame, (x1, y1), (x2, y2), color, 2)

                # Label text
                if tid in final_labels:
                    text = str(final_labels[tid])
                else:
                    text = f"ID {tid}"

                # Draw text
                cv2.putText(
                    draw_frame,
                    text,
                    (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    color,
                    2
                )

            last_drawn = draw_frame

        if last_drawn is None:
            continue

        if video_writer is None:
            h, w = last_drawn.shape[:2]
            video_writer = cv2.VideoWriter(
                str(output_path),
                cv2.VideoWriter_fourcc(*"mp4v"),
                30,
                (w, h)
            )

        video_writer.write(last_drawn)

    cap.release()

    if video_writer:
        video_writer.release()

    print("Pipeline complete. Video saved to:", output_path)

    return {
    "tracking": botsort_results,
    "ocr_output": output,
    "final_labels": final_labels,
    "track_alliance": track_alliance   
}

In [13]:
# RUN
output_dir = REPO_ROOT / "tracking_output"
output_dir.mkdir(exist_ok=True)

BLUE_IDS = [5809, 9570, 3928]
RED_IDS = [8825, 1736, 2357]

botsort_results = run_full_pipeline( 
    VIDEO_PATH,
    ROBOT_MODEL_PATH,
    NUMBER_MODEL_PATH,
    CUSTOM_TRACKER_PATH,
    BLUE_IDS,
    RED_IDS,
    output_path=output_dir / "final_output_with_reef.mp4",
    tracking_stride=3,
    ocr_stride=15
)


Running BotSort...
Running BotSort tracking...
WARNING ⚠️ not enough matching points
WARNING ⚠️ not enough matching points
Tracking complete.
Reading numbers...


TypeError: list indices must be integers or slices, not str

In [ ]:
from collections import defaultdict
import numpy as np

def build_frame_data(tracking_results, final_labels, track_alliance, reef_detections):
    
    frame_data = defaultdict(lambda: {"robots": [], "reef": None})

    for entry in tracking_results:
        f = entry["frame"]
        tid = entry["track_id"]

        frame_data[f]["robots"].append({
            "track_id": tid,
            "team": final_labels.get(tid),        
            "color": track_alliance.get(tid),     
            "bbox": entry["box"]
        })

    for r in reef_detections:
        frame_data[r["frame"]]["reef"] = r["box"]

    return frame_data

In [ ]:
def get_scoring_events(output_df):
    events = []

    prev_blue = 0
    prev_red = 0

    for _, row in output_df.iterrows():
        f = int(row["Frame"])

        if row["blue_score"] > prev_blue:
            events.append({
                "frame": f,
                "alliance": "blue",
                "points": row["blue_score"] - prev_blue
            })

        if row["red_score"] > prev_red:
            events.append({
                "frame": f,
                "alliance": "red",
                "points": row["red_score"] - prev_red
            })

        prev_blue = row["blue_score"]
        prev_red = row["red_score"]

    return events

In [ ]:
def compute_points(frame_data, scoring_events):

    from collections import defaultdict
    import numpy as np

    def center(box):
        x1, y1, x2, y2 = box
        return np.array([(x1+x2)/2, (y1+y2)/2])

    def dist(a, b):
        return np.linalg.norm(a - b)

    robot_points = defaultdict(int)
    SEARCH_WINDOW = 10

    for event in scoring_events:

        best_team = None
        best_dist = float("inf")

        for f in range(event["frame"] - SEARCH_WINDOW, event["frame"] + 1):

            if f not in frame_data:
                continue

            reef = frame_data[f]["reef"]
            if reef is None:
                continue

            reef_c = center(reef)

            for r in frame_data[f]["robots"]:

                if r["color"] != event["alliance"]:
                    continue

                if r["team"] is None:
                    continue

                d = dist(center(r["bbox"]), reef_c)

                if d < best_dist:
                    best_dist = d
                    best_team = r["team"]

        if best_team:
            robot_points[best_team] += event["points"]

    return robot_points

In [ ]:
def classify_roles(frame_data, robot_points):

    import numpy as np
    from collections import defaultdict

    def center(box):
        x1, y1, x2, y2 = box
        return np.array([(x1+x2)/2, (y1+y2)/2])

    def dist(a, b):
        return np.linalg.norm(a - b)

    robot_distances = defaultdict(list)

    for f, data in frame_data.items():

        if data["reef"] is None:
            continue

        reef_c = center(data["reef"])

        for r in data["robots"]:
            if r["team"] is None:
                continue

            d = dist(center(r["bbox"]), reef_c)
            robot_distances[r["team"]].append(d)

    roles = {}

    for team in robot_points:

        avg_dist = np.mean(robot_distances[team]) if robot_distances[team] else 999

        if robot_points[team] >= 5 and avg_dist < 200:
            roles[team] = "offense"
        else:
            roles[team] = "defense"

    return roles

In [ ]:
tracking = result["tracking"]
final_labels = result["final_labels"]
track_alliance = result["track_alliance"]

frame_data = build_frame_data(
    tracking,
    final_labels,
    track_alliance,
    reef_detections   
)

events = get_scoring_events(output_df)

robot_points = compute_points(frame_data, events)

roles = classify_roles(frame_data, robot_points)

df = build_results(robot_points, roles)

print(df)